# 🎯 TASK 5: Model Evaluation & Hyperparameter Tuning
## Going Beyond Accuracy - Precision, Recall, F1-Score & Optimization

---

## SETUP: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ Libraries imported')

---

## STEP 1: Load Cleaned Titanic Data

In [ ]:
from google.colab import files

print('Upload cleaned titanic.csv:')
uploaded = files.upload()

df = pd.read_csv('titanic.csv')

print('\n' + '='*80)
print('DATA LOADED')
print('='*80)
print(f'Shape: {df.shape}')
print(f'\nSurvival distribution:')
print(df['Survived'].value_counts())
print(f'\nClass balance:')
print(df['Survived'].value_counts(normalize=True) * 100)

---

## STEP 2: Prepare Data for Modeling

In [ ]:
# Ensure data is clean
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, errors='ignore')

# Encode categorical
df_encoded = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Remove non-numeric
X = df_encoded.drop('Survived', axis=1)
X = X.select_dtypes(include=[np.number])
y = df_encoded['Survived']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nNo missing values: {X.isnull().sum().sum()} ✓')

---

## STEP 3: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Train-Test Split:')
print(f'Training: {len(X_train)} samples')
print(f'Testing: {len(X_test)} samples')
print(f'\nTarget distribution in train: {y_train.value_counts().to_dict()}')
print(f'Target distribution in test: {y_test.value_counts().to_dict()}')

---

## STEP 4: Train Baseline Model

In [ ]:
print('\n' + '='*80)
print('BASELINE MODEL (Default Settings)')
print('='*80)

baseline_model = LogisticRegression(random_state=42, max_iter=1000)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)

print('\n✅ Baseline model trained')

---

## STEP 5: Evaluate Baseline - Accuracy vs Precision/Recall

In [ ]:
print('\n' + '='*80)
print('BASELINE MODEL EVALUATION')
print('='*80)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_precision = precision_score(y_test, y_pred_baseline)
baseline_recall = recall_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)

print(f'\nAccuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)')
print(f'Precision: {baseline_precision:.4f} ({baseline_precision*100:.2f}%)')
print(f'Recall: {baseline_recall:.4f} ({baseline_recall*100:.2f}%)')
print(f'F1-Score: {baseline_f1:.4f}')

print(f'\n' + '-'*80)
print('CLASSIFICATION REPORT:')
print('-'*80)
print(classification_report(y_test, y_pred_baseline, target_names=['Not Survived', 'Survived']))

---

## STEP 6: Why Accuracy Alone Can Be MISLEADING

### The Problem with Imbalanced Data:

In Titanic dataset: 62% died, 38% survived

Imagine a "lazy" model that always predicts "Did NOT Survive":
- Accuracy = 62% (correct on all non-survivors)
- Precision = NA (never predicts survival)
- Recall = 0% (misses ALL survivors)

This model is TERRIBLE at finding survivors, but accuracy looks decent!

### Real-World Examples:
- **Fraud Detection**: 99% legitimate transactions. Model always predicts "not fraud" = 99% accuracy but catches 0% fraud!
- **Disease Detection**: 95% healthy. Model always says "healthy" = 95% accuracy but kills patients by missing disease!
- **Titanic Survival**: 62% died. Predicting all deaths = 62% accuracy but wrong on survivors!

### Better Metrics:
- **Precision**: Of predicted survivors, how many actually survived? (False positive rate)
- **Recall**: Of actual survivors, how many did we find? (False negative rate)  
- **F1-Score**: Balance of precision & recall (harmonic mean)

### Why This Matters:
- Accuracy: Tells if model is right overall
- Precision: How trustworthy are positive predictions?
- Recall: How many positives did we catch?
- F1: Balanced score when data is imbalanced

In [ ]:
print('\n' + '='*80)
print('DEMONSTRATING IMBALANCED DATA PROBLEM')
print('='*80)

# Simulate lazy model that always predicts 0 (not survived)
y_pred_lazy = np.zeros(len(y_test))

lazy_accuracy = accuracy_score(y_test, y_pred_lazy)
lazy_precision = precision_score(y_test, y_pred_lazy, zero_division=0)
lazy_recall = recall_score(y_test, y_pred_lazy, zero_division=0)
lazy_f1 = f1_score(y_test, y_pred_lazy, zero_division=0)

print(f'\nLazy Model (Always Predicts "Not Survived"):')
print(f'Accuracy: {lazy_accuracy:.4f} ({lazy_accuracy*100:.2f}%)')
print(f'Precision: {lazy_precision:.4f}')
print(f'Recall: {lazy_recall:.4f}')
print(f'F1-Score: {lazy_f1:.4f}')

print(f'\n⚠️ See the problem?')
print(f'Accuracy looks decent ({lazy_accuracy*100:.1f}%) but Recall = 0 (finds ZERO survivors)')
print(f'\nThis is why we need Precision, Recall, and F1-Score!')

# Compare
comparison = pd.DataFrame({
    'Model': ['Lazy (Always 0)', 'Our Model'],
    'Accuracy': [f'{lazy_accuracy:.4f}', f'{baseline_accuracy:.4f}'],
    'Precision': [f'{lazy_precision:.4f}', f'{baseline_precision:.4f}'],
    'Recall': [f'{lazy_recall:.4f}', f'{baseline_recall:.4f}'],
    'F1-Score': [f'{lazy_f1:.4f}', f'{baseline_f1:.4f}']
})
print(f'\n{comparison.to_string(index=False)}')

---

## STEP 7: Hyperparameter Tuning with GridSearchCV

In [ ]:
print('\n' + '='*80)
print('HYPERPARAMETER TUNING - GridSearchCV')
print('='*80)

# Define parameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'max_iter': [500, 1000, 2000, 5000],
    'solver': ['lbfgs', 'liblinear']
}

print(f'\nSearching {len(param_grid["C"]) * len(param_grid["max_iter"]) * len(param_grid["solver"])} combinations...')
print(f'C values: {param_grid["C"]}')
print(f'max_iter values: {param_grid["max_iter"]}')
print(f'Solver values: {param_grid["solver"]}')

# GridSearchCV
grid_search = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)

print('\nTraining GridSearchCV (this may take a moment)...')
grid_search.fit(X_train, y_train)

print(f'\n✅ GridSearchCV complete!')
print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV F1-Score: {grid_search.best_score_:.4f}')

tuned_model = grid_search.best_estimator_

---

## STEP 8: Evaluate Tuned Model

In [ ]:
print('\n' + '='*80)
print('TUNED MODEL EVALUATION')
print('='*80)

y_pred_tuned = tuned_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

print(f'\nAccuracy: {tuned_accuracy:.4f} ({tuned_accuracy*100:.2f}%)')
print(f'Precision: {tuned_precision:.4f} ({tuned_precision*100:.2f}%)')
print(f'Recall: {tuned_recall:.4f} ({tuned_recall*100:.2f}%)')
print(f'F1-Score: {tuned_f1:.4f}')

print(f'\n' + '-'*80)
print('CLASSIFICATION REPORT:')
print('-'*80)
print(classification_report(y_test, y_pred_tuned, target_names=['Not Survived', 'Survived']))

---

## STEP 9: Before & After Comparison

In [ ]:
print('\n' + '='*80)
print('BEFORE vs AFTER TUNING')
print('='*80)

comparison_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Baseline': [
        f'{baseline_accuracy:.4f}',
        f'{baseline_precision:.4f}',
        f'{baseline_recall:.4f}',
        f'{baseline_f1:.4f}'
    ],
    'Tuned': [
        f'{tuned_accuracy:.4f}',
        f'{tuned_precision:.4f}',
        f'{tuned_recall:.4f}',
        f'{tuned_f1:.4f}'
    ],
    'Improvement': [
        f'{(tuned_accuracy - baseline_accuracy)*100:+.2f}%',
        f'{(tuned_precision - baseline_precision)*100:+.2f}%',
        f'{(tuned_recall - baseline_recall)*100:+.2f}%',
        f'{(tuned_f1 - baseline_f1):+.4f}'
    ]
})

print(f'\n{comparison_table.to_string(index=False)}')

print(f'\n' + '-'*80)
print('INTERPRETATION:')
print('-'*80)
if tuned_f1 > baseline_f1:
    print(f'✅ TUNING IMPROVED MODEL')
    print(f'   F1-Score increased by {(tuned_f1 - baseline_f1)*100:.2f}%')
else:
    print(f'⚠️ TUNING DID NOT IMPROVE MODEL')
    print(f'   F1-Score decreased by {(baseline_f1 - tuned_f1)*100:.2f}%')
    print(f'   Baseline hyperparameters were already good!')

---

## STEP 10: Confusion Matrix Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrix - Baseline vs Tuned Model', fontsize=14, fontweight='bold')

# Baseline
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', cbar=True, ax=axes[0],
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])
axes[0].set_title(f'Baseline (F1={baseline_f1:.3f})', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Tuned
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens', cbar=True, ax=axes[1],
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])
axes[1].set_title(f'Tuned (F1={tuned_f1:.3f})', fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

print('✅ Confusion matrices displayed')

---

## STEP 11: Metrics Comparison Chart

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
baseline_scores = [baseline_accuracy, baseline_precision, baseline_recall, baseline_f1]
tuned_scores = [tuned_accuracy, tuned_precision, tuned_recall, tuned_f1]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline', color='#3498db', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, tuned_scores, width, label='Tuned', color='#2ecc71', alpha=0.8, edgecolor='black')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison - Baseline vs Tuned\n', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=11)
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print('✅ Metrics comparison chart displayed')

---

## STEP 12: Feature Importance & Coefficients

In [ ]:
# Get top 10 features by importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Baseline_Coef': baseline_model.coef_[0],
    'Tuned_Coef': tuned_model.coef_[0]
}).copy()

feature_importance['Abs_Baseline'] = np.abs(feature_importance['Baseline_Coef'])
feature_importance['Abs_Tuned'] = np.abs(feature_importance['Tuned_Coef'])

top_features_baseline = feature_importance.nlargest(8, 'Abs_Baseline')
top_features_tuned = feature_importance.nlargest(8, 'Abs_Tuned')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 8 Features - Baseline vs Tuned Model', fontsize=14, fontweight='bold')

# Baseline
colors_base = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_features_baseline['Baseline_Coef']]
axes[0].barh(top_features_baseline['Feature'], top_features_baseline['Baseline_Coef'], 
             color=colors_base, alpha=0.8, edgecolor='black')
axes[0].set_title('Baseline Model', fontweight='bold')
axes[0].set_xlabel('Coefficient Value')
axes[0].grid(True, alpha=0.3, axis='x')

# Tuned
colors_tuned = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_features_tuned['Tuned_Coef']]
axes[1].barh(top_features_tuned['Feature'], top_features_tuned['Tuned_Coef'], 
             color=colors_tuned, alpha=0.8, edgecolor='black')
axes[1].set_title('Tuned Model', fontweight='bold')
axes[1].set_xlabel('Coefficient Value')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print('✅ Feature importance displayed')

---

## STEP 13: Final Summary

In [ ]:
print('\n' + '='*80)
print('🎉 TASK 5 COMPLETE - MODEL EVALUATION & TUNING')
print('='*80)

print(f'\n📊 BASELINE MODEL')
print(f'Accuracy: {baseline_accuracy:.4f}')
print(f'Precision: {baseline_precision:.4f}')
print(f'Recall: {baseline_recall:.4f}')
print(f'F1-Score: {baseline_f1:.4f}')

print(f'\n🎯 TUNED MODEL')
print(f'Accuracy: {tuned_accuracy:.4f}')
print(f'Precision: {tuned_precision:.4f}')
print(f'Recall: {tuned_recall:.4f}')
print(f'F1-Score: {tuned_f1:.4f}')

print(f'\n📈 IMPROVEMENTS')
print(f'Accuracy: {(tuned_accuracy - baseline_accuracy)*100:+.2f}%')
print(f'Precision: {(tuned_precision - baseline_precision)*100:+.2f}%')
print(f'Recall: {(tuned_recall - baseline_recall)*100:+.2f}%')
print(f'F1-Score: {(tuned_f1 - baseline_f1):+.4f}')

print(f'\n🔧 BEST HYPERPARAMETERS')
for param, value in grid_search.best_params_.items():
    print(f'{param}: {value}')

print(f'\n💡 KEY LESSONS')
print(f'✓ Accuracy alone can be misleading (especially imbalanced data)')
print(f'✓ Precision: How trustworthy are positive predictions?')
print(f'✓ Recall: How many positives did we catch?')
print(f'✓ F1-Score: Balance of precision and recall')
print(f'✓ Hyperparameter tuning systematically improves models')
print(f'✓ GridSearchCV tests multiple combinations')

print(f'\n✅ TASK 5 COMPLETE!')
print('='*80)